<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/Restart-From-Hackation_v1.0/mnps_post_getting_started%20v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MNPS Job Equity Post Mini-Hackathon
> A notebook to help you get started  
> DSI DSSG + MNPS Hackathon  
> September 9, 2025  
> Drafted by Wayne Birch - [contact her](wayne.birch@mnps.org) for questions, code update needs, or other questions about the notebook!

This notebook is a restart point based on the work done in the mini Hackathon with Metro Nashville Public Schools (MNPS) and the VU Data Science Institute (VU DSI).

 **Competition Details from the Hackathon with some updates follow:**

You aren't constrained to what is in this notebook, and please feel free to use your creativity to deliver the best solution
# **1** | Competition Parameters
* **Outcome and evaluation:** Participants will be evaluated on the performance of their provided solution on the holdout set. Importantly, judges must be able to easily run the submitted code on the new dataset.
* **Objective:** The overall objective is to create a system which best automatically, reproducibly, and reliably categorizes jobs according to the parameters set forth by MNPS. A few suggestions are provided on parameters that you can vary if you're thinking about achievable changes in 2.5 hours



## **2** | Environment Setup
Again, you're completely free to just download this notebook, create a local virtual environment and get to coding in your favorite IDE. We provide this code just as a rapid method to get started, and focus our efforts on implementation through Google Colab.

### **2a** | API Key Setup
#### **2a.1** | Access
The DSI has provided you an API key which can access **some** of the OpenAI models. These include:
* All versions of gpt-4o
* All versions of gpt-4.1
* All versions of o3-mini

Vector store upload, web search, code interpreter, and other functionality outside of the Chat Completions and Messages API is **not** supported. If you really want to use these things, you will have to make a good and cost-supported argument. If you don't feel like arguing, you can also utilize your own OpenAI API key.

#### **2a.2** | API Keys in Google Colab
To use your API key, click on the key icon (looks sort of like 🔑) in the left sidebar.  Under **Name**, add `OPENAI_API_KEY`. Under **Value**, paste your API key. Your API key is a jumble of numbers and letters, maybe even other symbols. Click the slider checkbox to enable **Notebook access** (so your notebook will grab these values without asking you).  

### **2b** | Runtime setup
We're going to install some packages in your environment so that you have access to the code functionality. If you need more packages, install more packages. Install **only** packages you trust.

In [ ]:
#Cell 3
!pip install -U openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 951.0/951.0 kB 14.0 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.106.0
    Uninstalling openai-1.106.0:
      Successfully uninstalled openai-1.106.0


In [ ]:
#Cell 3.25
!pip -q install -U openai reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 15.4 MB/s eta 0:00:00


In [ ]:
#Cell 3.5
# ===== Environment Setup (single source of truth) =====
import os
from typing import List
import pandas as pd
from pydantic import BaseModel, Field
from google.colab import userdata

# 1) API key from Colab's 🔑 panel
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 2) Read the model selector from Colab's 🔑 panel (can be alias or snapshot)
RAW_MODEL = userdata.get("OPENAI_MODEL")  # e.g., gpt-4o, gpt-4o-2024-11-20, gpt4.1, o3 mini

def normalize_model_id(s: str | None) -> str | None:
    if not s:
        return None
    s = s.strip().lower().replace("_", "-").replace(" ", "-")
    fixes = {
        "gpt4o": "gpt-4o",
        "gpt-4o": "gpt-4o",
        "gpt4.1": "gpt-4.1",
        "gpt-41": "gpt-4.1",
        "o3mini": "o3-mini",
        "o3-mini": "o3-mini",
    }
    return fixes.get(s, s)

alias_or_snapshot = normalize_model_id(RAW_MODEL)

# 3) Map aliases → pinned snapshots you prefer (edit to taste)
SNAPSHOTS = {
    # GPT-4o snapshots (stable; good for Structured Outputs)
    "gpt-4o":  "gpt-4o-2024-11-20",
    # GPT-4.1 family snapshot (long context)
    "gpt-4.1": "gpt-4.1-2025-04-14",
    # Keep o3-mini as an alias (no public dated snapshot ID); good for reasoning
    "o3-mini": "o3-mini",
}

# 4) Final MODEL_ID selection rule:
#    - If user entered an alias, pin it via SNAPSHOTS
#    - If user entered a snapshot, pass it through
#    - Else fallback to a safe default snapshot
MODEL_ID = SNAPSHOTS.get(alias_or_snapshot or "", None) or (alias_or_snapshot) or "gpt-4o-2024-11-20"

print("🔧 OPENAI_MODEL (raw):", RAW_MODEL)
print("✅ Using MODEL_ID:", MODEL_ID)


🔧 OPENAI_MODEL (raw): GPT-4o
✅ Using MODEL_ID: gpt-4o-2024-11-20


In [ ]:
# ===== Cell 4 — Unique run folder + get inputs (3 files) + robust CSV read + upload to OpenAI =====
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
from google.colab import drive
from openai import OpenAI

# ---------- 0) Mount Drive ----------
drive.mount('/content/drive')

# ---------- 1) Fixed output location (as requested) ----------
RUN_ROOT = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
timestamp = dt.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
RUN_DIR = RUN_ROOT / f"RUN_{timestamp}"
INPUTS_DIR = RUN_DIR / "inputs"
OUTPUTS_DIR = RUN_DIR / "outputs"
for p in (RUN_DIR, INPUTS_DIR, OUTPUTS_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("🗂️ Run folder:", RUN_DIR)

# ---------- 2) Where to find your three inputs by default ----------
# If you want to upload instead of copying from Drive, set ALLOW_UPLOAD = True.
DATA_INPUTS_DIR = Path("/content/drive/My Drive/Colab Notebooks/Data Inputs")
ALLOW_UPLOAD = False  # set True to be prompted to upload the 3 files from your computer

REQUIRED = {
    "Ground Truth Masterfile.csv": DATA_INPUTS_DIR / "Ground Truth Masterfile.csv",
    "New Sample_08.07.2025.csv":  DATA_INPUTS_DIR / "New Sample_08.07.2025.csv",
    "MNPS_Prompt_Resources.zip":  DATA_INPUTS_DIR / "MNPS_Prompt_Resources.zip",
}

# (A) Optionally upload files instead of copying from Drive
if ALLOW_UPLOAD:
    from google.colab import files as colab_files
    print("🔼 Upload the three files when prompted:")
    uploaded = colab_files.upload()  # opens a browser picker
    for name in REQUIRED.keys():
        if name in uploaded:
            dst = INPUTS_DIR / name
            with open(dst, "wb") as f:
                f.write(uploaded[name])
            REQUIRED[name] = dst  # point to the just-uploaded copy

# (B) Copy from Drive if not already present in /inputs
missing = []
for name, src in REQUIRED.items():
    dst = INPUTS_DIR / name
    if dst.exists():
        continue
    if src.exists():
        shutil.copy2(src, dst)
        print(f"📄 Copied: {src}  →  {dst}")
    else:
        missing.append(name)

if missing:
    raise FileNotFoundError(
        "These input files were not found. Place them in "
        f"{DATA_INPUTS_DIR} or enable ALLOW_UPLOAD=True:\n - " + "\n - ".join(missing)
    )

# ---------- 3) Unpack the resources zip into inputs/resources (optional but helpful) ----------
resources_zip = INPUTS_DIR / "MNPS_Prompt_Resources.zip"
RESOURCES_DIR = INPUTS_DIR / "resources"
if resources_zip.exists():
    RESOURCES_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(resources_zip, "r") as zf:
        zf.extractall(RESOURCES_DIR)
    print("🧰 Unpacked resources to:", RESOURCES_DIR)

# ---------- 4) Robust CSV reader (handles cp1252/latin1) ----------
def read_csv_smart(path: Path, **kwargs) -> pd.DataFrame:
    trials = [
        dict(encoding="utf-8"),
        dict(encoding="utf-8-sig"),
        dict(encoding="cp1252"),
        dict(encoding="latin1"),
    ]
    for t in trials:
        try:
            df = pd.read_csv(path, **{**t, **kwargs})
            print(f"✅ Read {path.name} with encoding={t['encoding']}")
            return df
        except UnicodeDecodeError:
            continue
    # last resort
    df = pd.read_csv(path, encoding="latin1", on_bad_lines="skip", **kwargs)
    print(f"⚠️ Read {path.name} with encoding=latin1 (on_bad_lines='skip')")
    return df

# Smoke test: load one row from the sample CSV (row 0) and build job_desc_text for downstream cells
sample_csv = INPUTS_DIR / "New Sample_08.07.2025.csv"
df = read_csv_smart(sample_csv)

required_cols = [
    "Job Description Name","Position Summary","Education","Work Experience",
    "Essential Functions","Licenses and Certifications","Knowledge, Skills and Abilities"
]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in {sample_csv.name}: {missing_cols}")

ROW_IDX = 0
r = df.iloc[ROW_IDX]
job_desc_text = f"""Job Description Name: {r['Job Description Name']}

Position Summary: {r['Position Summary']}
Education: {r['Education']}
Work Experience: {r['Work Experience']}
Licenses and Certifications: {r['Licenses and Certifications']}
Essential Functions: {r['Essential Functions']}
Knowledge, Skills and Abilities: {r['Knowledge, Skills and Abilities']}
"""
print("🧪 Prepared job_desc_text from row", ROW_IDX)

# ---------- 5) Upload the two CSVs to OpenAI so later cells can attach them ----------
client = OpenAI()  # API key already set in your Environment Setup cell
to_upload = [
    INPUTS_DIR / "Ground Truth Masterfile.csv",
    INPUTS_DIR / "New Sample_08.07.2025.csv",
]
uploaded = []
for p in to_upload:
    with open(p, "rb") as f:
        up = client.files.create(file=f, purpose="assistants")
    uploaded.append(up)

file_ids = [u.id for u in uploaded]  # <-- used by the Responses API cell later
print("⬆️ Uploaded file_ids:", file_ids)

# ---------- 6) Write a small manifest so you can audit each run ----------
manifest = {
    "run_folder": str(RUN_DIR),
    "created_utc": timestamp,
    "inputs": [str(p) for p in (INPUTS_DIR / "Ground Truth Masterfile.csv",
                                 INPUTS_DIR / "New Sample_08.07.2025.csv")],
    "resources_dir": str(RESOURCES_DIR) if RESOURCES_DIR.exists() else None,
    "uploaded_file_ids": file_ids,
}
(RUN_DIR / "RUN_METADATA.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("\n📁 Current run tree (first few entries):")
for i, p in enumerate(sorted(RUN_DIR.rglob("*"))):
    print(" -", p.relative_to(RUN_DIR))
    if i > 25:
        print(" … (truncated)")
        break

Mounted at /content/drive


/tmp/ipython-input-419050840.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = dt.datetime.utcnow().strftime("%Y%m%d_%H%M%S")


🗂️ Run folder: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250909_211428
📄 Copied: /content/drive/My Drive/Colab Notebooks/Data Inputs/Ground Truth Masterfile.csv  →  /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250909_211428/inputs/Ground Truth Masterfile.csv
📄 Copied: /content/drive/My Drive/Colab Notebooks/Data Inputs/New Sample_08.07.2025.csv  →  /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250909_211428/inputs/New Sample_08.07.2025.csv
📄 Copied: /content/drive/My Drive/Colab Notebooks/Data Inputs/MNPS_Prompt_Resources.zip  →  /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250909_211428/inputs/MNPS_Prompt_Resources.zip
🧰 Unpacked resources to: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250909_211428/inputs/resources
✅ Read New Sample_08.07.2025.csv with encoding=cp1252
🧪 Prepared job_desc_text from row 0
⬆️ Uploaded file_ids: ['file-7WUyLDTQ3fhN4StJLg8j6m', 'file-Csam8hRZuoe2iCDJhoAufL']

📁 Current run tree (first fe

In [ ]:
# Cell 6
from openai import OpenAI
client = OpenAI()

visible = {m.id for m in client.models.list().data}
if MODEL_ID not in visible:
    print(f"⚠️ {MODEL_ID} is not visible to your key. "
          "Use an alias you do see (e.g., gpt-4o) or confirm access in your org.")
else:
    print(f"👍 {MODEL_ID} is available.")


👍 gpt-4o-2024-11-20 is available.


## **3** | The Data

The current prompt is a two-step prompt that is successful through the ChatGPT interface. It requires two types of data:
* The data to be classified
* Supporting resources

We need to read all of this in. Let's grab it and use it. The first thing you'll do is just straight up download a zip file of all of this information.

You can download all of the reference files from the link provided, then upload in the sidebar. You'll then unzip the directory using the code below.

Click on the folder icon in the left sidebar (kinda looks like this 🗂️) and you'll see all the files there. We'll read them in.


In [ ]:
# Cell 8
from google.colab import drive
drive.mount('/content/drive')
base_target_folder = '/content/drive/My Drive/Colab Notebooks/Data Inputs'
!unzip "{base_target_folder}/MNPS_Prompt_Resources.zip" -d /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archive:  /content/drive/My Drive/Colab Notebooks/Data Inputs/MNPS_Prompt_Resources.zip
  inflating: /content/Korn_Ferry Lominger 38 Competencies.csv  
  inflating: /content/Competency Extended Descriptions.csv  
  inflating: /content/MNPS KSACs.csv  
  inflating: /content/MNPS Roles.csv  


## **4** | The Prompts

What we have here is a direct prompt to get the response that we're looking for. We'll make this happen directly using the OpenAI Chat Completions API. Note that you can use other APIs as you like.

In [ ]:
#Cell 12
zero_shot_prompt = \
""" Objective: Evaluate and group jobs from the "New Sample_08.07.2025.csv" file based on similarities in job functions, not job titles.

Process:

- Compare all jobs against each other using the attributes listed in the file: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.
- Compare each job with reference sources using the same attributes. I have attached the reference sources for you.
- Group jobs based on similarities into:
  - Major role groupings (e.g., Specialist, Analyst, Manager)
  - Minor sub-groupings (e.g., I, II, III, IV) - not to exceed level IV
- Use the MNPS Roles and MNPS KSACs documents to help you determine major role groupings.
- Use the remaining documents to help you clarify subtle differences in role groupings and sub-groupings.
- Use a more qualitative, holistic assessment focused on functional alignment with KSACs rather than a quantitative scoring approach with defined complexity metrics

Output Format:

- Create a table with the following columns:
  - Original Job Title
  - New Job Title
  - Major Role Group
  - Minor Sub-Group
  - Justification for Grouping

- Provide an accompanying narrative explaining the rationale behind the groupings and any notable patterns or insights discovered during the analysis.

Job Title Convention:

- Follow the format: "[Function] [Role] [Level]" (e.g., "Collections Specialist II", "Accounts Payable Specialist III")

Additional Guidelines:

- Ensure all sources used are cited properly.
- Focus on the nature of the work performed rather than just the job titles.
- Consider the complexity of tasks, level of responsibility, and required competencies when determining groupings.
- Provide clear explanations for why each job was classified as it was, referencing specific job attributes and external benchmarks.

"""

Instead of asking for a table output, we will use **structured outputs**. Though this is a common approach for the outputs of LLMs/AI systems, you can learn more about this on [OpenAI's structured output documentation](https://platform.openai.com/docs/guides/structured-outputs?api-mode=responses). Note that you can find this information on almost all LLM/AI platform or package providers.

In [ ]:
#Cell 14
from pydantic import BaseModel, Field

class JobClassification(BaseModel):
    """Represents the classification of a job based on its functions."""
    job_title_original: str = Field(..., description="The original job title as provided in the input data using the job title convention specified.")
    new_job_title: str = Field(..., description="The proposed new job title based on the classification using the job title convention specified.")
    major_role_group: str = Field(..., description="The major grouping of the job based on its functional role (e.g., Specialist, Analyst, Manager).")
    minor_sub_group: str = Field(..., description="The minor sub-grouping within the major role group (e.g., Specialist I, II, III, IV).")
    grouping_justification: str = Field(..., description="The justification for placing the job in the specific major and minor groups, referencing job attributes and relevant documents.")

In [ ]:
#Cell 15
from typing import List

class JobClassificationTable(BaseModel):
  """The table classification and overall commentary on the groupings provided by the AI system."""
  job_classification_table: List[JobClassification] = Field(..., description="The table of job classifications.")
  narrative_rationale: str = Field(..., description="The narrative commentary on the groupings provided by the AI system.")

Create classifications using OpenAI. Of note here is:
* The **developer** prompt - this is the "system prompt" or "custom instructions" for the model. This determines the overall behavior of the model.
* The **user** prompt - this is what we send to the model like when we're chatting with ChatGPT.

In [ ]:
# ===== Cell 16 — Responses API (TEXT-ONLY, no attachments), saves to OUTPUTS_DIR =====
from openai import OpenAI
from pathlib import Path
import pandas as pd, json, inspect

client = OpenAI()  # API key from Environment Setup

# ---- Requires earlier cells ----
assert 'MODEL_ID' in globals(), "Run Environment Setup first (MODEL_ID)."
assert 'INPUTS_DIR' in globals() and 'OUTPUTS_DIR' in globals(), "Run the unique-run Cell 4 first."
assert 'zero_shot_prompt' in globals(), "Define zero_shot_prompt in your prompt cell."
assert 'job_desc_text' in globals(), "Cell 4 builds job_desc_text (row 0 smoke test)."

print("🤖 Using model:", MODEL_ID)

# If read_csv_smart exists (Cell 4), use it for robust encodings; else default to utf-8
def _read_csv(path: Path, **kw):
    if 'read_csv_smart' in globals():
        return read_csv_smart(path, **kw)
    return pd.read_csv(path, encoding="utf-8", **kw)

# ---- Build a SMALL context from Ground Truth (first 3 rows) ----
gt_path = Path(INPUTS_DIR) / "Ground Truth Masterfile.csv"
context_block = ""
if gt_path.exists():
    try:
        gt_df = _read_csv(gt_path).fillna("")
        # keep only lightweight columns if present
        preferred_cols = [
            "Original Job Title","New Job Title","Major Role Group","Minor Sub-Group","Justification for Grouping",
            "Position Summary","Education","Work Experience","Licenses and Certifications","Essential Functions","Knowledge, Skills and Abilities"
        ]
        cols = [c for c in preferred_cols if c in gt_df.columns] or list(gt_df.columns)[:8]
        mini = gt_df[cols].head(3)
        # represent as compact JSON so the model can parse easily
        context_block = "Context (Ground Truth examples):\n" + mini.to_json(orient="records", force_ascii=False)
    except Exception as e:
        context_block = f"Context note: Ground Truth CSV present but could not be summarized ({e})."

# ---- Compose text-only input (no attachments) ----
# Tip: the zero_shot_prompt you wrote mentions "attached reference sources" — we add a Context block instead.
full_text = (
    zero_shot_prompt.strip()
    + "\n\n"
    + (context_block + "\n\n" if context_block else "")
    + "Classify the following job description:\n\n"
    + job_desc_text
)

# ---- Capability detection for your SDK version ----
def _has_param(obj, name: str) -> bool:
    try:
        return name in inspect.signature(obj).parameters
    except Exception:
        return False

supports_parse_schema  = _has_param(client.responses.parse,  "response_format")
supports_create_schema = _has_param(client.responses.create, "response_format")

parsed = None
raw_text = ""

try:
    if supports_parse_schema:
        # Newer SDK: server-enforced Structured Outputs via parse()
        resp = client.responses.parse(
            model=MODEL_ID,
            input=[{"role": "user", "content": [{"type":"input_text","text": full_text}]}],
            temperature=0.2,
            max_output_tokens=1400,
            response_format=JobClassificationTable,  # Pydantic schema (Cells 14–15)
        )
        parsed  = resp.output_parsed
        raw_text = resp.output_text or ""
    elif supports_create_schema:
        # Mid SDK: enforce via create() + json_schema
        schema = JobClassificationTable.model_json_schema()
        resp = client.responses.create(
            model=MODEL_ID,
            input=[{"role": "user", "content": [{"type":"input_text","text": full_text}]}],
            temperature=0.2,
            max_output_tokens=1400,
            response_format={
                "type": "json_schema",
                "json_schema": {"name": "JobClassificationTable", "schema": schema, "strict": True},
            },
        )
        raw_text = getattr(resp, "output_text", None) or ""
        # Clean the raw_text to remove potential markdown formatting
        if raw_text.strip().startswith("```json"):
            raw_text = raw_text.strip()[7:].strip("`")
        parsed = JobClassificationTable.model_validate_json(raw_text) if raw_text else None
    else:
        # Old SDK: prompt-only enforcement + client-side validation
        schema_json = json.dumps(JobClassificationTable.model_json_schema(), indent=2)
        strict_text = (
            "You MUST return ONLY valid JSON that matches the following JSON Schema. No prose, no markdown.\n"
            "JSON Schema:\n" + schema_json + "\n\n"
            "Task:\n" + full_text
        )
        resp = client.responses.create(
            model=MODEL_ID,
            input=[{"role": "user", "content": [{"type":"input_text","text": strict_text}]}],
            temperature=0.2,
            max_output_tokens=1400,
        )
        raw_text = getattr(resp, "output_text", None) or ""
        # Clean the raw_text to remove potential markdown formatting
        if raw_text.strip().startswith("```json"):
            raw_text = raw_text.strip()[7:].strip("`")
        parsed = JobClassificationTable.model_validate_json(raw_text) if raw_text else None
except Exception as e:
    print("❗ Unexpected Responses API error:", e)
    raise

# ---- Save outputs to OUTPUTS_DIR ----
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
raw_path = Path(OUTPUTS_DIR) / "Raw_Response.json"
raw_path.write_text(raw_text or "", encoding="utf-8")

if parsed is not None:
    rows = [row.model_dump() for row in parsed.job_classification_table]
    out_csv = Path(OUTPUTS_DIR) / "Job_Classifications.csv"
    pd.DataFrame(rows).to_csv(out_csv, index=False, encoding="utf-8")
    out_txt = Path(OUTPUTS_DIR) / "Narrative.txt"
    out_txt.write_text(parsed.narrative_rationale, encoding="utf-8")
    print("✅ Saved:", out_csv)
    print("✅ Saved:", out_txt)
else:
    print("⚠️ No parsed object returned; saved Raw_Response.json only at:", raw_path)

print("✅ Saved:", raw_path)

# ---- Console visibility ----
print("\n=== RAW JSON STRING FROM MODEL ===")
print(raw_text or "(empty)")
if parsed is not None:
    print("\n=== PARSED (Pydantic) ===")
    print(parsed.model_dump_json(indent=2))

# ---- List run outputs ----
print("\nContents of OUTPUTS_DIR:")
for p in sorted(Path(OUTPUTS_DIR).glob("*")):
    print(" -", p.name)

🤖 Using model: gpt-4o-2024-11-20
✅ Read Ground Truth Masterfile.csv with encoding=cp1252
✅ Saved: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250909_211428/outputs/Job_Classifications.csv
✅ Saved: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250909_211428/outputs/Narrative.txt
✅ Saved: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250909_211428/outputs/Raw_Response.json

=== RAW JSON STRING FROM MODEL ===

{
  "job_classification_table": [
    {
      "job_title_original": "Tech Mail Center",
      "new_job_title": "Mail Center Technician I",
      "major_role_group": "Technician",
      "minor_sub_group": "I",
      "grouping_justification": "The role involves operational and logistical tasks such as sorting, delivering, and managing mail, which aligns with the Technician major role group. The minor sub-group 'I' is assigned due to the entry-level nature of the position, as indicated by the required High School Diploma, basic mail certification, 

In [ ]:
# ===== Cell 16.5 — FAST batch (TEXT-ONLY) with progress, timeouts, checkpoints =====
from openai import OpenAI
from pathlib import Path
import pandas as pd, json, time, random, inspect

# ---- prerequisites from earlier cells ----
assert 'df' in globals(), "Run Cell 4 first (loads df & job_desc_text helper)."
assert 'OUTPUTS_DIR' in globals(), "Run the unique-run cell first (defines OUTPUTS_DIR)."
assert 'MODEL_ID' in globals(), "Run Environment Setup first (defines MODEL_ID)."
assert 'zero_shot_prompt' in globals(), "Define zero_shot_prompt (your prompt cell)."

# Use the same robust CSV reader if it's defined
def _read_csv(path: Path, **kw):
    if 'read_csv_smart' in globals():
        return read_csv_smart(path, **kw)
    return pd.read_csv(path, encoding="utf-8", **kw)

# Build a tiny context from Ground Truth (first 3 rows), once
gt_path = Path(INPUTS_DIR) / "Ground Truth Masterfile.csv" if 'INPUTS_DIR' in globals() else None
context_block = ""
if gt_path and gt_path.exists():
    try:
        gt_df = _read_csv(gt_path).fillna("")
        preferred_cols = [
            "Original Job Title","New Job Title","Major Role Group","Minor Sub-Group","Justification for Grouping",
            "Position Summary","Education","Work Experience","Licenses and Certifications","Essential Functions","Knowledge, Skills and Abilities"
        ]
        cols = [c for c in preferred_cols if c in gt_df.columns] or list(gt_df.columns)[:8]
        mini = gt_df[cols].head(3)
        context_block = "Context (3 ground-truth examples):\n" + mini.to_json(orient="records", force_ascii=False)
    except Exception as e:
        context_block = f"(Context unavailable: {e})"

# Helper to build the job text block for a row
def build_job_text(r):
    return f"""Job Description Name: {r.get('Job Description Name', '')}

Position Summary: {r.get('Position Summary', '')}
Education: {r.get('Education', '')}
Work Experience: {r.get('Work Experience', '')}
Licenses and Certifications: {r.get('Licenses and Certifications', '')}
Essential Functions: {r.get('Essential Functions', '')}
Knowledge, Skills and Abilities: {r.get('Knowledge, Skills and Abilities', '')}
"""

# Compose the full prompt for a row
def full_text_for_row(r):
    return (
        zero_shot_prompt.strip()
        + ("\n\n" + context_block if context_block else "")
        + "\n\nClassify the following job description:\n\n"
        + build_job_text(r)
    )

# ---- OpenAI client with timeouts/retries ----
client = OpenAI(timeout=60.0, max_retries=2)  # 60s per request; adjust as needed

# Capability detection for your SDK
def _has_param(obj, name: str) -> bool:
    try:
        return name in inspect.signature(obj).parameters
    except Exception:
        return False
supports_parse_schema  = _has_param(client.responses.parse,  "response_format")
supports_create_schema = _has_param(client.responses.create, "response_format")

# ---- batching parameters (start small, then increase) ----
ROW_START   = 0
ROW_LIMIT   = 10          # ← start with 10; set to None when you're happy with speed
TEMP        = 0.2
MAX_TOKENS  = 700         # smaller = faster/cheaper; raise if you need longer narratives
SAVE_EVERY  = 5           # checkpoint every N rows
MAX_ATTEMPTS_PER_ROW = 3  # retries (light)

records = []
errors  = []

def call_model_with_text(text):
    if supports_parse_schema:
        resp = client.responses.parse(
            model=MODEL_ID,
            input=[{"role": "user", "content": [{"type":"input_text","text": text}]}],
            temperature=TEMP,
            max_output_tokens=MAX_TOKENS,
            response_format=JobClassificationTable,  # Pydantic
        )
        return resp.output_parsed, resp.output_text or ""
    elif supports_create_schema:
        schema = JobClassificationTable.model_json_schema()
        resp = client.responses.create(
            model=MODEL_ID,
            input=[{"role": "user", "content": [{"type":"input_text","text": text}]}],
            temperature=TEMP,
            max_output_tokens=MAX_TOKENS,
            response_format={
                "type":"json_schema",
                "json_schema":{"name":"JobClassificationTable","schema":schema,"strict":True},
            },
        )
        raw = getattr(resp, "output_text", None) or ""
        return JobClassificationTable.model_validate_json(raw), raw
    else:
        # Old SDK: prompt-only + client-side validate
        schema_json = json.dumps(JobClassificationTable.model_json_schema(), indent=2)
        strict = (
            "You MUST return ONLY valid JSON that matches the following JSON Schema. No prose, no markdown.\n"
            f"JSON Schema:\n{schema_json}\n\nTask:\n{text}"
        )
        resp = client.responses.create(
            model=MODEL_ID,
            input=[{"role":"user","content":[{"type":"input_text","text": strict}]}],
            temperature=TEMP,
            max_output_tokens=MAX_TOKENS,
        )
        raw = getattr(resp, "output_text", None) or ""
        return JobClassificationTable.model_validate_json(raw), raw

def backoff_sleep(k):
    time.sleep(min(20, 1.8**k + random.random()))

# ---- iterate rows with progress logging ----
start_time = time.time()
end_idx = len(df) if ROW_LIMIT is None else min(len(df), ROW_START + ROW_LIMIT)

for i in range(ROW_START, end_idx):
    r = df.iloc[i]
    text = full_text_for_row(r)

    t0 = time.time()
    parsed = None
    raw    = ""

    for attempt in range(MAX_ATTEMPTS_PER_ROW):
        try:
            parsed, raw = call_model_with_text(text)
            break
        except Exception as e:
            msg = str(e)
            if "429" in msg or "rate" in msg.lower() or "timeout" in msg.lower():
                backoff_sleep(attempt)
                continue
            if attempt == MAX_ATTEMPTS_PER_ROW - 1:
                errors.append((i, msg[:500]))
            else:
                backoff_sleep(attempt)

    if parsed:
        for rec in parsed.job_classification_table:
            row_out = rec.model_dump()
            row_out["source_row_index"] = i
            row_out["model_used"] = MODEL_ID
            records.append(row_out)

    # checkpoint save
    if (i - ROW_START + 1) % SAVE_EVERY == 0 or i == end_idx - 1:
        if records:
            pd.DataFrame(records).to_csv(Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv", index=False, encoding="utf-8")
        Path(OUTPUTS_DIR, "Batch_Errors.json").write_text(json.dumps(errors, indent=2), encoding="utf-8")

    # progress line
    print(f"[{i+1}/{end_idx}] row done in {time.time()-t0:.1f}s | total {(time.time()-start_time)/60:.1f} min | "
          f"ok {len(records)} | err {len(errors)}")

print("✅ Batch done. Saved CSV/JSON to:", OUTPUTS_DIR)

#============== Make the batch file show up in the master even before housekeeping ====================

from pathlib import Path, shutil
src = Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv"
dst = Path(OUTPUTS_DIR) / "Job_Classifications.csv"
if src.exists():
    import shutil
    shutil.copy2(src, dst)
    print("📄 Copied batch to:", dst)


✅ Read Ground Truth Masterfile.csv with encoding=cp1252
[1/10] row done in 22.0s | total 0.4 min | ok 1 | err 0
[2/10] row done in 8.9s | total 0.5 min | ok 2 | err 0
[3/10] row done in 6.2s | total 0.6 min | ok 3 | err 0
[4/10] row done in 6.6s | total 0.7 min | ok 4 | err 0
[5/10] row done in 24.8s | total 1.1 min | ok 5 | err 0
[6/10] row done in 13.4s | total 1.4 min | ok 6 | err 0
[7/10] row done in 10.8s | total 1.5 min | ok 7 | err 0
[8/10] row done in 30.9s | total 2.1 min | ok 7 | err 1
[9/10] row done in 22.4s | total 2.4 min | ok 7 | err 2
[10/10] row done in 4.0s | total 2.5 min | ok 8 | err 2
✅ Batch done. Saved CSV/JSON to: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250909_211428/outputs


In [ ]:
# ===== Cell 16.6 — Quick sanity check for current run =====
from pathlib import Path
import pandas as pd, json

assert 'OUTPUTS_DIR' in globals(), "Run your unique-run cell first (defines OUTPUTS_DIR)."

batch = Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv"
if batch.exists():
    dfb = pd.read_csv(batch)
    print("✅ Batch rows in this run:", len(dfb))
    display(dfb.head(5))
else:
    print("⚠️ No batch file found at", batch)

errs = Path(OUTPUTS_DIR) / "Batch_Errors.json"
if errs.exists():
    e = json.loads(Path(errs).read_text())
    print("⚠️ Rows with errors:", len(e))
    if e:
        print("First error:", e[0])


In [ ]:
#Cell 17
# Inspect parsed output (Responses API)
try:
    parsed  # from Cell 16
    print(parsed.model_dump_json(indent=2))
except NameError:
    print("No 'parsed' object found. Run Cell 16 first.")


{
  "job_classification_table": [
    {
      "job_title_original": "Associate Consumer Insights Analyst",
      "new_job_title": "Consumer Insights Analyst I",
      "major_role_group": "Analyst",
      "minor_sub_group": "I",
      "grouping_justification": "The role focuses on analyzing consumer data, deriving actionable insights, and supporting marketing decisions, which aligns with the Analyst major role group. The responsibilities, such as using SQL for data analysis, creating visualizations, and collaborating with stakeholders, indicate an entry-level position within the Analyst group, justifying the 'I' sub-group classification."
    }
  ],
  "narrative_rationale": "The job was classified under the Analyst major role group due to its focus on data analysis, consumer insights, and collaboration with marketing teams to drive business decisions. The minor sub-group 'I' was assigned based on the entry-level nature of the role, as evidenced by the required 1-3 years of experience an

We can make this into a table using pandas!

In [ ]:
# Cell 17.5 — Build a response_dict from the Responses API parsed object
from pathlib import Path
import json
import pandas as pd

# Make sure Cell 16 ran (it defines `parsed`) and the run folders exist
assert 'parsed' in globals(), "Run Cell 16 first (it sets `parsed`)."
assert 'OUTPUTS_DIR' in globals(), "Run the unique-run cell first (defines OUTPUTS_DIR)."

# Convert the Pydantic objects to plain dicts
response_dict = {
    "job_classification_table": [rec.model_dump() for rec in parsed.job_classification_table],
    "narrative_rationale": parsed.narrative_rationale,
}

# Optional: preview the first rows
display(pd.DataFrame(response_dict["job_classification_table"]).head(10))

# Optional: save a pretty JSON alongside your other outputs
out_json = Path(OUTPUTS_DIR) / "Parsed_Response.json"
out_json.write_text(json.dumps(response_dict, indent=2), encoding="utf-8")
print("Saved:", out_json)

# Also return the dict so it shows below the cell
response_dict


,job_title_original,new_job_title,major_role_group,minor_sub_group,grouping_justification
0,Associate Consumer Insights Analyst,Consumer Insights Analyst I,Analyst,I,"The role focuses on analyzing consumer data, d..."


Saved: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250909_211428/outputs/Parsed_Response.json


{'job_classification_table': [{'job_title_original': 'Associate Consumer Insights Analyst',
   'new_job_title': 'Consumer Insights Analyst I',
   'major_role_group': 'Analyst',
   'minor_sub_group': 'I',
   'grouping_justification': "The role focuses on analyzing consumer data, deriving actionable insights, and supporting marketing decisions, which aligns with the Analyst major role group. The responsibilities, such as using SQL for data analysis, creating visualizations, and collaborating with stakeholders, indicate an entry-level position within the Analyst group, justifying the 'I' sub-group classification."}],
 'narrative_rationale': "The job was classified under the Analyst major role group due to its focus on data analysis, consumer insights, and collaboration with marketing teams to drive business decisions. The minor sub-group 'I' was assigned based on the entry-level nature of the role, as evidenced by the required 1-3 years of experience and the tasks performed, which involve

In [ ]:
#Cell 18
# Preview the saved classifications CSV (if present)
from pathlib import Path
import pandas as pd

csv_path = Path(OUTPUTS_DIR) / "Job_Classifications.csv"
if csv_path.exists():
    display(pd.read_csv(csv_path).head(10))
else:
    print("No Job_Classifications.csv found in", OUTPUTS_DIR)


,job_title_original,new_job_title,major_role_group,minor_sub_group,grouping_justification
0,Tech Mail Center,Mail Center Technician I,Technician,I,The role involves operational and logistical t...


In [ ]:
# Cell 19 ===== Housekeeping & Archive (Run Results) =====
# Place this cell at the END of the notebook. Run after your pipeline finishes.
from google.colab import drive
from pathlib import Path
import shutil, json, re
import datetime as dt
import pandas as pd

# ---------- CONFIG (edit to taste) ----------
RUN_ROOT = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
ARCHIVE_DIR = RUN_ROOT / "_archives"
MASTER_DIR  = RUN_ROOT / "_master"

KEEP_LAST_N_RUNS   = 10     # keep this many newest runs; older ones can be deleted
ZIP_OLDER_RUNS     = True   # zip runs (into _archives) to save space
PURGE_RAW_JSON     = True   # delete outputs/Raw_Response.json inside each run
PURGE_PARSED_JSON  = False  # delete outputs/Parsed_Response.json
PURGE_BATCH_ERRORS = False  # delete outputs/Batch_Errors.json
SKIP_CURRENT_RUN   = True   # don't zip/purge/delete the most recent run
DRY_RUN            = True   # <<< safety: set False to actually apply changes

# ---------- Mount Drive (no-op if already mounted) ----------
drive.mount('/content/drive')

# ---------- Helpers ----------
def parse_run_ts(name: str):
    m = re.match(r"RUN_(\d{8}_\d{6})$", name)
    if not m:
        return None
    try:
        return dt.datetime.strptime(m.group(1), "%Y%m%d_%H%M%S")
    except Exception:
        return None

def folder_size_bytes(p: Path) -> int:
    total = 0
    for f in p.rglob("*"):
        if f.is_file():
            try:
                total += f.stat().st_size
            except Exception:
                pass
    return total

def human_mb(nbytes: int) -> str:
    return f"{nbytes/1_000_000:.2f} MB"

# ---------- Discover run folders ----------
runs = []
for d in RUN_ROOT.iterdir():
    if d.is_dir() and d.name.startswith("RUN_"):
        ts = parse_run_ts(d.name)
        if ts:
            runs.append((d, ts))

runs.sort(key=lambda x: x[1], reverse=True)  # newest first
print(f"Found {len(runs)} run folders under: {RUN_ROOT}")

current = runs[0][0] if runs else None
if current:
    print("Most recent run:", current.name)

# Summary of the first few
for d, ts in runs[:5]:
    print(f" - {d.name} | {ts:%Y-%m-%d %H:%M:%S} | size≈ {human_mb(folder_size_bytes(d))}")

# Ensure archive/master dirs
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
MASTER_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Plan actions ----------
actions = []

# 1) Purge large intermediates within runs
def plan_purges(d: Path):
    out = d / "outputs"
    if not out.exists():
        return
    if PURGE_RAW_JSON and (out / "Raw_Response.json").exists():
        actions.append(("delete_file", out / "Raw_Response.json"))
    if PURGE_PARSED_JSON and (out / "Parsed_Response.json").exists():
        actions.append(("delete_file", out / "Parsed_Response.json"))
    if PURGE_BATCH_ERRORS and (out / "Batch_Errors.json").exists():
        actions.append(("delete_file", out / "Batch_Errors.json"))

# 2) Zip older runs (into _archives)
def plan_zip(d: Path):
    z = ARCHIVE_DIR / f"{d.name}.zip"
    if not z.exists():
        actions.append(("zip_folder", (d, z)))

# 3) Delete runs beyond retention
to_prune = runs[KEEP_LAST_N_RUNS:] if KEEP_LAST_N_RUNS is not None else []
for d, ts in runs:
    if SKIP_CURRENT_RUN and current and d == current:
        continue
    # Purges
    plan_purges(d)
    # Zip plan
    if ZIP_OLDER_RUNS:
        plan_zip(d)

for d, ts in to_prune:
    actions.append(("delete_folder", d))

# ---------- Show plan ----------
print("\nPlanned actions:")
if not actions:
    print(" (none)")
else:
    for act, obj in actions:
        if act == "zip_folder":
            d, z = obj
            print(f" - ZIP {d.name}  →  {z.name}")
        else:
            print(f" - {act.upper()}: {obj}")

# ---------- Execute (unless DRY_RUN) ----------
if DRY_RUN:
    print("\nDRY_RUN=True — no changes applied. Set DRY_RUN=False to execute.")
else:
    for act, obj in actions:
        try:
            if act == "delete_file":
                Path(obj).unlink(missing_ok=True)
            elif act == "zip_folder":
                d, z = obj
                # create zip in ARCHIVE_DIR; shutil.make_archive adds .zip automatically
                base_name = z.with_suffix("")  # remove .zip for make_archive
                shutil.make_archive(str(base_name), 'zip', root_dir=d)
            elif act == "delete_folder":
                shutil.rmtree(obj, ignore_errors=True)
        except Exception as e:
            print("  ! Error:", act, obj, e)
    print("\n✅ Housekeeping complete.")

# ---------- Aggregate a master CSV across all runs (safe to do anytime) ----------
frames = []
for d, ts in runs:
    for name in ["Job_Classifications_Batch.csv", "Job_Classifications.csv"]:
        csvp = d / "outputs" / name
        meta = d / "RUN_METADATA.json"
        if csvp.exists():
            try:
                df_run = pd.read_csv(csvp)
                df_run["run_folder"]  = d.name
                df_run["source_file"] = name
                # enrich with metadata if available
                if meta.exists():
                    try:
                        m = json.loads(meta.read_text())
                        df_run["created_utc"] = m.get("created_utc")
                        df_run["model_used"]  = m.get("resolved_model_id") or m.get("model_used")
                    except Exception:
                        pass
                frames.append(df_run)
            except Exception as e:
                print(f"  ! Skipping {csvp.name} due to read error:", e)

if frames:
    master = pd.concat(frames, ignore_index=True)
    MASTER_DIR.mkdir(parents=True, exist_ok=True)
    master_out = MASTER_DIR / "All_Job_Classifications.csv"
    master.to_csv(master_out, index=False, encoding="utf-8")
    print(f"\n📚 Master CSV updated: {master_out} ({len(master)} rows; from {len(frames)} files)")
else:
    print("\n(No job classification CSVs found to aggregate.)")
